# Lab 14: Segmentation

**Module 14** | Read `notes/14-segmentation-unet-maskrcnn.pdf` before running this notebook.

Run a pre-trained DeepLabV3 on a sample image and visualize semantic segmentation masks.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


## Semantic segmentation with DeepLabV3

DeepLabV3 combines atrous convolutions and a lightweight decoder head to assign a class label to every pixel. The ResNet-50 variant pre-trained on COCO outputs 21 Pascal-VOC-style classes (background + 20 objects).

Unlike detection (boxes), segmentation produces a dense label map. We colorize that map and display it beside the original image.


### Load model and sample image

Torchvision weights include the exact preprocessing (resize + ImageNet normalization) the network expects.


In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from runtime_env import configure_runtime

configure_runtime(quiet=True)

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from torchvision.models.segmentation import (
    DeepLabV3_ResNet50_Weights,
    deeplabv3_resnet50,
)
from runtime_env import ensure_sample_images

IMAGE_DIR = ensure_sample_images()

weights = DeepLabV3_ResNet50_Weights.DEFAULT
model = deeplabv3_resnet50(weights=weights)
model.eval().to(device)
preprocess = weights.transforms()

image_paths = sorted(IMAGE_DIR.glob("*.jpg"))
if not image_paths:
    raise FileNotFoundError(
        f"No JPG in {IMAGE_DIR}. Check your network connection or run: python download_datasets.py"
    )
img_path = image_paths[0]
original = Image.open(img_path).convert("RGB")
print(f"Input image: {img_path.name} ({original.size[0]}x{original.size[1]})")


### Run inference

The output `out["out"]` has shape `[1, num_classes, H, W]`. Taking `argmax` over the class dimension yields the per-pixel label.


In [ ]:
batch = preprocess(original).unsqueeze(0).to(device)
with torch.no_grad():
    out = model(batch)["out"]

mask = out.argmax(dim=1).squeeze(0).cpu().numpy()
print(f"Segmentation map shape: {mask.shape}")
print(f"Unique classes in scene: {np.unique(mask).tolist()}")


### Colorize and compare

We map each class id to a distinct RGB color (deterministic via a fixed seed) and show input versus colored mask side by side.


In [ ]:
rng = np.random.default_rng(42)
num_classes = out.shape[1]
palette = rng.integers(0, 255, size=(num_classes, 3), dtype=np.uint8)
palette[0] = [0, 0, 0]  # background stays black

colored = palette[mask]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(original)
axes[0].set_title("Input")
axes[0].axis("off")

axes[1].imshow(colored)
axes[1].set_title("DeepLabV3 segmentation (colored classes)")
axes[1].axis("off")

plt.tight_layout()
plt.show()
